In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
np.random.seed(109204)
plt.rcParams.update({'axes.labelsize': 18, 'xtick.labelsize': 16, 'ytick.labelsize': 16, 'axes.titlesize': 18,})

def henon_map(x, y, a, b): # Function for Henon
    return 1 - a*x**2 + y, b*x


    
sigma_Q = 0.5  # Controls Variability of Q_k
def ekf_henon_estimation(a_true=1.4, b_true=0.3, a_guess=1, b_guess=0.1, n_steps=200, Q_error=1e-5, x0_true=0.1, y0_true=0.1, x0_hat=0.1, y0_hat=0.1, meas_noise=None):
    Q0 = Q_error
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps))
    generated = False
    if meas_noise is None:
        generated = True
        x_true = np.zeros(n_steps)
        y_true = np.zeros(n_steps)
        x_meas = np.zeros(n_steps)
        y_meas = np.zeros(n_steps)
        x_true[0] = x0_true
        y_true[0] = y0_true
        x_meas[0] = x_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        y_meas[0] = y_true[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
        for k in range(1, n_steps):
            x_true[k], y_true[k] = henon_map(x_true[k-1], y_true[k-1], a_true, b_true)  
            x_meas[k] = x_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
            y_meas[k] = y_true[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    else:
        x_true = None # If data does already exist, just use that
        y_true = None

    x_hat = np.zeros(n_steps) # EKF Setup
    y_hat = np.zeros(n_steps)
    a_hat = np.zeros(n_steps)
    b_hat = np.zeros(n_steps)

    x_hat[0] = x0_hat
    y_hat[0] = y0_hat
    a_hat[0] = a_guess    
    b_hat[0] = b_guess
    P = np.eye(4)

    C = np.array([ # Measurements for both x and y
        [1, 0, 0, 0],
        [0, 1, 0, 0]])

    for k in range(1, n_steps):
        x_prev, y_prev, a_prev, b_prev = x_hat[k-1], y_hat[k-1], a_hat[k-1], b_hat[k-1]

        x_pred, y_pred = henon_map(x_prev, y_prev, a_prev, b_prev) # Prediction Stage
        a_pred, b_pred = a_prev, b_prev
        s_pred = np.array([[x_pred], [y_pred], [a_pred], [b_pred]])

        Phi = np.array([
            [-2*a_prev*x_prev, 1, -x_prev**2, 0], # Jacobian
            [b_prev, 0, 0, x_prev],
            [0, 0, 1, 0],
            [0, 0, 0, 1]])
        P_pred = Phi @ P @ Phi.T

        z = np.array([[x_meas[k]], [y_meas[k]]]) # Update Measurement
        z_pred = C @ s_pred
        innovation = z - z_pred

        S = C @ P_pred @ C.T + Q_seq[k] * np.eye(2)
        K = P_pred @ C.T @ np.linalg.inv(S)

        s_upd = s_pred + K @ innovation
        P = (np.eye(4) - K @ C) @ P_pred # Non-Optimal Update (not Joseph Form)

        s_upd = np.asarray(s_upd).reshape(-1)
        x_hat[k], y_hat[k], a_hat[k], b_hat[k] = s_upd[0], s_upd[1], s_upd[2], s_upd[3]
    return x_true, y_true, x_meas, y_meas, x_hat, y_hat, a_hat, b_hat

def interactive_ekf_henon(a_true=1.4, b_true=0.3, Q_error=1e-5, n_steps=200):
    plt.close('all')
    a_guess, b_guess = 1, 0.1
    n_runs = 100

    Q0 = Q_error
    Q_seq = Q0 * np.exp(sigma_Q * np.random.randn(n_steps)) 
    x_true_base = np.zeros(n_steps) # True Henon Map (Single Trajectory)
    y_true_base = np.zeros(n_steps)
    x_meas_base = np.zeros(n_steps)
    y_meas_base = np.zeros(n_steps)
    x_true_base[0] = 0.7
    y_true_base[0] = 0.7
    x_meas_base[0] = x_true_base[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
    y_meas_base[0] = y_true_base[0] + np.random.normal(0, np.sqrt(Q_seq[0]))
    for k in range(1, n_steps):
        x_true_base[k], y_true_base[k] = henon_map(x_true_base[k-1], y_true_base[k-1], a_true, b_true)
        x_meas_base[k] = x_true_base[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
        y_meas_base[k] = y_true_base[k] + np.random.normal(0, np.sqrt(Q_seq[k]))
    
    all_x_true = np.zeros((n_runs, n_steps))
    all_y_true = np.zeros((n_runs, n_steps))
    all_x_hat = np.zeros((n_runs, n_steps))
    all_y_hat = np.zeros((n_runs, n_steps))
    all_a_hat = np.zeros((n_runs, n_steps))
    all_b_hat = np.zeros((n_runs, n_steps))

    for i in range(n_runs):
        x_true, y_true, x_meas, y_meas, x_hat, y_hat, a_hat, b_hat = ekf_henon_estimation(
            a_true=a_true,
            b_true=b_true,
            a_guess=a_guess,
            b_guess=b_guess,
            n_steps=n_steps,
            Q_error=Q_error, 
            meas_noise = None)
        
        all_x_true[i, :] = x_true_base
        all_y_true[i, :] = y_true_base
        all_x_hat[i, :] = x_hat
        all_y_hat[i, :] = y_hat
        all_a_hat[i, :] = a_hat
        all_b_hat[i, :] = b_hat

    x_hat_mean = np.mean(all_x_hat, axis=0) # Statistics from MC
    y_hat_mean = np.mean(all_y_hat, axis=0)
    x_hat_p05 = np.percentile(all_x_hat, 5, axis=0)
    x_hat_p95 = np.percentile(all_x_hat, 95, axis=0)
    y_hat_p05 = np.percentile(all_y_hat, 5, axis=0)
    y_hat_p95 = np.percentile(all_y_hat, 95, axis=0)

    mse_x = np.mean((all_x_hat - all_x_true)**2, axis=0) # MSE of MC
    variance_x = np.var(all_x_hat, axis=0) # Filter Spread (MC Variance)
    mse_y = np.mean((all_y_hat - all_y_true)**2, axis=0)
    variance_y = np.var(all_y_hat, axis=0)

    a_hat_mean = np.mean(all_a_hat, axis=0)
    b_hat_mean = np.mean(all_b_hat, axis=0)
    a_hat_p05 = np.percentile(all_a_hat, 5, axis=0)
    a_hat_p95 = np.percentile(all_a_hat, 95, axis=0)
    b_hat_p05 = np.percentile(all_b_hat, 5, axis=0)
    b_hat_p95 = np.percentile(all_b_hat, 95, axis=0)    

    fig, ax = plt.subplots(2, 4, figsize=(18, 7), sharex=True) # Array Plot

    ax[0, 0].plot(x_true_base, 'k', lw=2, label='True Henon Map for x') # For x
    ax[0, 0].plot(x_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 0].plot(x_meas_base, 'r.', alpha=0.3, label='Measurements')
    ax[0, 0].set_xlabel('Iteration') 
    ax[0, 0].set_title('x_k evolution')
    ax[0, 0].legend()

    ax[0, 1].plot(y_true_base, 'k', lw=2, label='True Henon Map for y') # For y
    ax[0, 1].plot(y_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 1].plot(y_meas_base, 'r.', alpha=0.3, label='Measurements')
    ax[0, 1].set_xlabel('Iteration') 
    ax[0, 1].set_title('y_k evolution')
    ax[0, 1].legend()

    ax[0, 2].axhline(a_true, color='k', lw=2, label='True a') # For a
    ax[0, 2].plot(a_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 2].set_title('EKF a Evolution Across MC Runs')
    ax[0, 2].set_xlabel('Iteration')
    ax[0, 2].set_ylabel('a')
    ax[0, 2].legend()

    ax[0, 3].axhline(b_true, color='k', lw=2, label='True b') # For b
    ax[0, 3].plot(b_hat_mean, 'b--', label='Mean EKF Estimate')
    ax[0, 3].set_title('EKF b Evolution Across MC Runs')
    ax[0, 3].set_xlabel('Iteration')
    ax[0, 3].set_ylabel('b')
    ax[0, 3].legend()

    ax[1, 0].plot(mse_x, 'b-', label='MSE') # MC Evolution x
    ax[1, 0].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 0].set_xlabel('Iteration')
    ax[1, 0].set_ylabel('MSE')
    ax[1, 0].set_title('MSE Evolution of x Over Time')
    ax[1, 0].legend()

    ax[1, 1].plot(mse_y, 'b-', label='MSE') # MC Evolution y
    ax[1, 1].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 1].set_xlabel('Iteration')
    ax[1, 1].set_ylabel('MSE')
    ax[1, 1].set_title('MSE Evolution of y Over Time')
    ax[1, 1].legend()

    ax[1, 2].plot(variance_x, 'r-', label='MC Variance') # MC Variance x
    ax[1, 2].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 2].set_xlabel('Iteration')
    ax[1, 2].set_ylabel('Variance')
    ax[1, 2].set_title('Filter Spread of MC in x')
    ax[1, 2].legend()

    ax[1, 3].plot(variance_y, 'r-', label='MC Variance') # MC Variance y
    ax[1, 3].axhline(0, color='k', linestyle='--', linewidth=1)
    ax[1, 3].set_xlabel('Iteration')
    ax[1, 3].set_ylabel('Variance')
    ax[1, 3].set_title('Filter Spread of MC in y')
    ax[1, 3].legend()

    fig.suptitle(
        f"EKF for Henon Map, with a_true = {a_true:.2f}, b_true = {b_true:.2f}, Q = {Q_error:.1e}",
        fontsize=16)    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

interact( # Interactive Sliders
    interactive_ekf_henon,
    a_true=FloatSlider(value=1.4, min=0, max=1.42, step=0.01), # Max before Infinite Divergence
    b_true=FloatSlider(value=0.3, min=-0.3, max=0.31, step=0.01),
    Q_error=FloatSlider(value=1e-5, min=1e-12, max=1e-2,
                  step=1e-6, readout_format='.1e'),
    n_steps=IntSlider(value=200, min=100, max=500, step=50))

interactive(children=(FloatSlider(value=1.4, description='a_true', max=1.42, step=0.01), FloatSlider(value=0.3…

<function __main__.interactive_ekf_henon(a_true=1.4, b_true=0.3, Q_error=1e-05, n_steps=200)>